# E1 — Huấn luyện YOLOv10n baseline

Notebook này chạy protocol E1 của project: YOLOv10n pretrained, kích thước ảnh 640, seed 42, và không thêm các cải tiến riêng của E2–E5. Nó gọi trực tiếp các hàm trong `src/helmet_yolov10`, nên hành vi giống CLI `scripts/train.py`.

## Chuẩn bị một lần

Từ thư mục gốc repository, tạo môi trường và mở Jupyter bằng `uv`:

```bash
uv sync --extra dev
uv pip install git+https://github.com/THU-MIG/yolov10.git
uv run --with jupyterlab jupyter lab
```

Trước khi train, cần có dataset theo `configs/data.yaml` và weight tại `weights/yolov10n.pt`. Cell train mặc định không chạy; chỉ đổi `RUN_TRAIN = True` sau khi các cell kiểm tra đã thành công.

In [ ]:
from __future__ import annotations

import os  òioijdsoigjsoidgj
import sys
from pathlib import Path
from pprint import pprint

# Notebook may be opened from notebooks/ or from another Jupyter working directory.
candidate = Path.cwd().resolve()
for directory in (candidate, *candidate.parents):
    if (directory / 'pyproject.toml').is_file():
        PROJECT_ROOT = directory
        break
else:
    raise RuntimeError('Không tìm thấy project root (pyproject.toml).')

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

E1_CONFIG = PROJECT_ROOT / 'configs' / 'E1_baseline.yaml'
DATA_CONFIG = PROJECT_ROOT / 'configs' / 'data.yaml'
WEIGHTS = PROJECT_ROOT / 'weights' / 'yolov10n.pt'

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')

## 1. Xác nhận cấu hình E1 đã được resolve

`E1_baseline.yaml` kế thừa các tham số model/train/evaluation từ `base.yaml`.

In [ ]:
from helmet_yolov10.utils.config import load_config

config = load_config(E1_CONFIG)
assert config['experiment']['id'] == 'E1'
assert config['model']['architecture'].lower() == 'yolov10n'
assert config['training']['imgsz'] == 640

pprint({
    'experiment': config['experiment'],
    'model': config['model'],
    'training': {key: config['training'][key] for key in ('data', 'imgsz', 'epochs', 'batch', 'seed', 'device')},
    'output': config['output'],
})

## 2. Kiểm tra dataset YOLO

Validator kiểm tra cả ba split `train`, `val`, `test`; ảnh/label tương ứng; class ID; bounding box chuẩn hoá; và rò rỉ giữa split. Sửa `configs/data.yaml` hoặc chuẩn bị dataset trước khi tiếp tục nếu cell này báo lỗi.

In [ ]:
from helmet_yolov10.data.validation import validate_dataset

dataset_report = validate_dataset(DATA_CONFIG, PROJECT_ROOT)
pprint(dataset_report)

## 3. Kiểm tra pretrained weight và backend

Không cần tải model ở cell này. Nó chỉ xác nhận weight và package backend đã sẵn sàng.

In [ ]:
from helmet_yolov10.training.train import _load_backend
from helmet_yolov10.utils.environment import sha256_file

if not WEIGHTS.is_file():
    raise FileNotFoundError(
        f'Không tìm thấy {WEIGHTS}. Tải yolov10n.pt và đặt vào weights/.'
    )

backend_class = _load_backend()
print(f'Backend: {backend_class}')
print(f'Weights: {WEIGHTS}')
print(f'SHA-256: {sha256_file(WEIGHTS)}')

## 4. Dry run — kiểm tra toàn bộ input của E1

Cell này chạy cùng validation với train thật nhưng không load model, không dùng GPU và không tạo experiment run.

In [ ]:
from helmet_yolov10.training.train import train_baseline

train_baseline(E1_CONFIG, validate_only=True)
print('E1 dry run thành công — dataset và config đã hợp lệ.')

## 5. Train E1

Đặt `RUN_TRAIN = True` để thực sự train. Nếu GPU khác, đổi `DEVICE`; dùng `'cpu'` chỉ để thử nghiệm vì sẽ chậm. Tên run phải mới, vì trainer sẽ không ghi đè run cũ.

In [ ]:
RUN_TRAIN = False  # Đổi thành True sau khi đã hoàn tất các kiểm tra ở trên.
DEVICE = 0
RUN_NAME = 'baseline_seed42'

if RUN_TRAIN:
    run_dir = train_baseline(
        E1_CONFIG,
        run_name=RUN_NAME,
        device=DEVICE,
    )
    print(f'Train hoàn tất: {run_dir}')
else:
    print('Chưa train. Đổi RUN_TRAIN thành True rồi chạy lại cell này.')

## 6. Đánh giá `best.pt` trên test set

Chỉ chạy bước này sau khi train thành công. Kết quả metrics và metadata được lưu trong `experiments/E1/evaluation/<tên-run>/`.

In [ ]:
from helmet_yolov10.evaluation.evaluate import evaluate_checkpoint

RUN_EVALUATION = False  # Đổi thành True sau khi best.pt đã tồn tại.
CHECKPOINT = PROJECT_ROOT / 'experiments' / 'E1' / RUN_NAME / 'weights' / 'best.pt'
EVALUATION_RUN_NAME = f'{RUN_NAME}_test'

if RUN_EVALUATION:
    if not CHECKPOINT.is_file():
        raise FileNotFoundError(f'Không tìm thấy checkpoint: {CHECKPOINT}')
    evaluation_dir = evaluate_checkpoint(
        CHECKPOINT,
        E1_CONFIG,
        run_name=EVALUATION_RUN_NAME,
        device=DEVICE,
    )
    print(f'Đánh giá hoàn tất: {evaluation_dir}')
else:
    print('Chưa đánh giá. Đổi RUN_EVALUATION thành True rồi chạy lại cell này.')

## 7. Đọc metrics đã lưu

Sau khi evaluation hoàn tất, cell dưới đây in các metric do backend YOLO trả về.

In [ ]:
import json

metrics_path = (
    PROJECT_ROOT / 'experiments' / 'E1' / 'evaluation' / EVALUATION_RUN_NAME / 'metrics.json'
)
if metrics_path.is_file():
    with metrics_path.open(encoding='utf-8') as handle:
        pprint(json.load(handle))
else:
    print(f'Chưa có metrics tại: {metrics_path}')